In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 72.8 MB/s eta 0:00:00


In [9]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# 🚀 Force Streamlit Light Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#ffd803"\nbackgroundColor="#f9fcfc"\nsecondaryBackgroundColor="#e3f6f5"\ntextColor="#2d334a"\n')

st.set_page_config(page_title="Infosys Portal", page_icon="⚡", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "bg_main": "#f9fcfc", "bg_sidebar": "#e3f6f5", "bg_card": "#ffffff", "bg_card_alt": "#bae8e8",
    "text_main": "#2d334a", "text_heading": "#272343", "text_muted": "#64748b",
    "accent": "#ffd803", "accent_hover": "#e6c300", "accent_text": "#272343",
    "border": "#272343", "border_light": "#bae8e8", "success": "#34d399", "danger": "#f87171"
}

JWT_SECRET = JWT_SECRET = os.environ.get("JWT_SECRET")

# 🚀 Neo-Brutalist CSS + Visible Sidebar Re-Open Button
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');
    html, body, .stApp {{ background: {COLORS['bg_main']} !important; font-family: 'Inter', sans-serif !important; color: {COLORS['text_main']} !important; }}

    /* 🚀 DO NOT HIDE HEADER! Only hide footer and bottom decoration so toggle arrow stays visible! */
    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    /* 🚀 Highlight Streamlit's Native Sidebar Re-Open Button (Top-Left) as a Yellow Badge! */
    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['accent']} !important; border: 2px solid {COLORS['border']} !important;
        border-radius: 8px !important; padding: 6px !important; margin: 8px !important;
        box-shadow: 3px 3px 0px {COLORS['border']} !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['text_heading']} !important; color: {COLORS['text_heading']} !important; stroke: {COLORS['text_heading']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_heading']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; }}

    /* Inputs */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"] {{ background-color: {COLORS['bg_card']} !important; border: 2px solid {COLORS['border']} !important; border-radius: 10px !important; }}
    div[data-baseweb="input"]:focus-within {{ border-color: {COLORS['accent']} !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; }}
    input, div[data-baseweb="select"] span {{ color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important; }}

    /* Buttons */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['accent']} !important; color: {COLORS['accent_text']} !important;
        border: 2px solid {COLORS['border']} !important; border-radius: 10px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 700 !important; font-size: 14px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; box-shadow: 4px 4px 0px {COLORS['border']} !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent_hover']} !important; transform: translate(-2px, -2px) !important;
        box-shadow: 6px 6px 0px {COLORS['border']} !important;
    }}
    section[data-testid="stSidebar"] {{ background: {COLORS['bg_sidebar']} !important; border-right: 2px solid {COLORS['border']} !important; }}
    .pn-card {{ background: {COLORS['bg_card']}; border: 2px solid {COLORS['border']}; border-radius: 14px; padding: 24px; box-shadow: 4px 4px 0px {COLORS['border_light']}; }}
</style>
""", unsafe_allow_html=True)

def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Intelligent Analytics Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.5rem 0 1rem;">
        <div style="font-size:40px;margin-bottom:10px;">⚡</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:14px;margin:4px 0 0;">{sub}</p>
    </div>
    <div style="text-align:center;margin-bottom:1.5rem;"><span style="font-size:1.1rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                if r and check_txt(pwd, r[0]): st.session_state.token = make_jwt(email); navigate("Dashboard")
                else: st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname or not email or len(pwd) < 8 or not sa:
                    st.error("⚠️ Please fill all fields (password min 8 chars).")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")
            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                    if r:
                        st.session_state.reset_email = email
                        st.session_state.sq_p = r[0]
                        st.session_state.reset_mode = "sq"
                        st.rerun()
                    else: st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    pass
            else:
                if st.session_state.get("reset_mode") == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password (min 8 chars)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Reset Password →", use_container_width=True):
                        if len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated successfully!"); time.sleep(1); st.session_state.reset_email = None; navigate("Login")
                            else: st.error("❌ Incorrect security answer.")
            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    # 🚀 SIDEBAR MENU
    with st.sidebar:
        st.markdown(f"""
        <div style="padding:16px 8px;text-align:center;">
            <div style="font-size:28px;">⚡</div>
            <div style="font-weight:700;font-size:16px;color:{COLORS['text_heading']};">Infosys Portal</div>
            <div style="font-size:11px;color:{COLORS['text_muted']};">{"Admin Panel" if email=="infosys@ai" else "Intelligent Analytics"}</div>
        </div><hr style="border-color:{COLORS['border_light']};">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={"container": {"background-color": COLORS['bg_sidebar']}, "nav-link-selected": {"background-color": COLORS['accent'], "color": COLORS['accent_text']}})
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    # 🚀 ADMIN DASHBOARD (infosys@ai)
    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:36px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

    # 🚀 REGULAR USER DASHBOARD
    else:
        st.markdown(f"""
        <div style="background:{COLORS['text_heading']};border-radius:16px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;">
            <div><h1 style="color:{COLORS['accent']} !important;margin:0;font-size:24px !important;">⚡ Infosys Portal</h1><div style="color:{COLORS['bg_card_alt']};font-size:13px;">Analytics Dashboard</div></div>
            <div style="background:{COLORS['accent']};padding:8px 18px;border-radius:30px;font-weight:700;color:{COLORS['text_heading']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:28px;">{icon}</div>
                <div style="font-size:26px;font-weight:700;color:{COLORS['text_heading']};">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['accent']}, "bgcolor": COLORS['bg_card_alt'], "borderwidth": 1, "bordercolor": COLORS['border']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)


Overwriting app.py


In [ ]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Retrieve your secret token securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")


🚀 Infosys Portal Live URL: https://figurine-stillness-hertz.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.


In [7]:

from google.colab import userdata

required_secrets = [
    "NGROK_AUTHTOKEN",
    "JWT_SECRET",
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD"
]

for key in required_secrets:
    value = userdata.get(key)
    print(f"{key}: {'Configured ✅' if value else 'MISSING ❌'}")


NGROK_AUTHTOKEN: Configured ✅
JWT_SECRET: Configured ✅
EMAIL_ADDRESS: Configured ✅
EMAIL_PASSWORD: Configured ✅


In [22]:

from pathlib import Path
import os
from google.colab import userdata

# 1. Load secrets into the notebook process environment.
# Values are never printed.
for key in ["JWT_SECRET", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]:
    value = userdata.get(key)
    if not value:
        raise RuntimeError(f"Missing Colab Secret: {key}")
    os.environ[key] = value

# 2. Read the app created by the original %%writefile cell.
app_path = Path("app.py")
if not app_path.exists():
    raise FileNotFoundError(
        "app.py not found. Run the original %%writefile app.py cell first."
    )

code = app_path.read_text()

# 3. Remove the hard-coded JWT signing secret.
old_jwt = 'JWT_SECRET = "super-secret-infosys-key-2026"'
new_jwt = '''JWT_SECRET = os.environ.get("JWT_SECRET")
if not JWT_SECRET:
    raise RuntimeError("JWT_SECRET is not configured.")'''

if old_jwt in code:
    code = code.replace(old_jwt, new_jwt, 1)
elif 'JWT_SECRET = os.environ.get("JWT_SECRET")' not in code:
    raise RuntimeError("Could not locate the expected JWT_SECRET line.")

# 4. Add email and password validators.
if "def valid_email(email):" not in code:
    anchor = "def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False"
    helpers = '''

def valid_email(email):
    # At least 2 letters before @, 2 in the domain,
    # and 2 after the final dot.
    return bool(re.fullmatch(
        r"[A-Za-z]{2,}[A-Za-z0-9._%+-]*@[A-Za-z0-9-]{2,}(?:\\.[A-Za-z0-9-]+)*\\.[A-Za-z]{2,}",
        email.strip()
    ))

def valid_password(password):
    return (
        len(password) >= 8
        and any(c.isupper() for c in password)
        and any(c.islower() for c in password)
        and any(c.isdigit() for c in password)
        and any(not c.isalnum() for c in password)
    )
'''
    if anchor not in code:
        raise RuntimeError("Could not find the helper-function insertion point.")
    code = code.replace(anchor, anchor + helpers, 1)

# Ensure re is imported.
if "import re" not in code:
    code = code.replace(
        "import os, sqlite3, jwt, bcrypt, datetime, time, streamlit as st",
        "import os, re, sqlite3, jwt, bcrypt, datetime, time, streamlit as st",
        1
    )

# 5. Strengthen signup validation.
old_signup = '''if not uname or not email or len(pwd) < 8 or not sa:
                    st.error("⚠️ Please fill all fields (password min 8 chars).")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")'''
new_signup = '''if not all([uname.strip(), email, pwd, confirm_pwd, sa.strip()]):
                    st.error("⚠️ Please fill in all required fields.")
                elif not valid_email(email):
                    st.error("❌ Enter a valid email address.")
                elif not valid_password(pwd):
                    st.error("⚠️ Password must be 8+ characters with uppercase, lowercase, number, and special symbol.")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")'''

if old_signup in code:
    code = code.replace(old_signup, new_signup, 1)
elif "not valid_password(pwd)" not in code:
    raise RuntimeError("Could not locate the expected signup validation block.")

# 6. After signup, return to Login. A JWT is issued only on login.
old_signup_success = '''                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")'''
new_signup_success = '''                        st.session_state.token = None
                        st.success("✅ Account created! Please sign in.")
                        time.sleep(1)
                        navigate("Login")'''

if old_signup_success in code:
    code = code.replace(old_signup_success, new_signup_success, 1)
elif "Please sign in." not in code:
    raise RuntimeError("Could not locate the signup success block.")

# 7. Apply the same strong password rule to security-question reset.
old_reset = '''if len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")
                        elif npw != confirm_npw:'''
new_reset = '''if not valid_password(npw):
                            st.error("⚠️ Password must be 8+ characters with uppercase, lowercase, number, and special symbol.")
                        elif npw != confirm_npw:'''

if old_reset in code:
    code = code.replace(old_reset, new_reset, 1)
elif "not valid_password(npw)" not in code:
    raise RuntimeError("Could not locate the expected password-reset validation block.")

# 8. Save the patched app.py.
app_path.write_text(code)

# 9. Syntax-check the updated Python file.
import ast
ast.parse(code)

print("✅ app.py updated successfully.")
print("✅ JWT secret now comes from Colab Secrets.")
print("✅ Email and strong-password validators added.")
print("✅ Signup now returns to Login.")
print("✅ Password-reset validation updated.")
print("✅ Python syntax check passed.")
print("🔒 Secret values were not displayed.")


KeyboardInterrupt: 

In [17]:

import re

def valid_email(email):
    return bool(re.fullmatch(
        r"[A-Za-z]{2,}[A-Za-z0-9._%+-]*@[A-Za-z0-9-]{2,}(?:\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}",
        email.strip()
    ))

def valid_password(password):
    return (
        len(password) >= 8
        and any(c.isupper() for c in password)
        and any(c.islower() for c in password)
        and any(c.isdigit() for c in password)
        and any(not c.isalnum() for c in password)
    )

print("Email ab@cd.ef:", valid_email("ab@cd.ef"))
print("Email a@cd.ef:", valid_email("a@cd.ef"))
print("Password Ankit@123:", valid_password("Ankit@123"))
print("Password ankit123:", valid_password("ankit123"))


Email ab@cd.ef: True
Email a@cd.ef: False
Password Ankit@123: True
Password ankit123: False


In [ ]:

from pathlib import Path
import ast

path = Path("app.py")
code = path.read_text()

# Add imports needed for generating and emailing OTPs.
if "import smtplib" not in code:
    code = code.replace(
        "import os, re, sqlite3, jwt, bcrypt, datetime, time, streamlit as st",
        "import os, re, secrets, smtplib, sqlite3, jwt, bcrypt, datetime, time, streamlit as st\nfrom email.message import EmailMessage",
        1
    )

# Add OTP session-state defaults.
old_state = 'for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None)]:'
new_state = 'for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None), ("otp_code", None), ("otp_expiry", None), ("otp_verified", False)]:'

if old_state in code:
    code = code.replace(old_state, new_state, 1)

# Replace the existing placeholder for the Via OTP button.
old_otp_button = '''                if col_otp.button("Via OTP", use_container_width=True):
                    pass'''

new_otp_button = '''                if col_otp.button("Via OTP", use_container_width=True):
                    if not email:
                        st.error("Enter your registered email address.")
                    elif not valid_email(email):
                        st.error("Enter a valid email address.")
                    else:
                        with get_db() as c:
                            user = c.execute(
                                "SELECT username FROM users WHERE email=?", (email,)
                            ).fetchone()

                        if not user:
                            st.error("If the account exists, an OTP can be sent to its registered email.")
                        else:
                            try:
                                otp = f"{secrets.randbelow(1000000):06d}"
                                msg = EmailMessage()
                                msg["Subject"] = "Infosys Portal - Password Reset OTP"
                                msg["From"] = os.environ["EMAIL_ADDRESS"]
                                msg["To"] = email
                                msg.set_content(
                                    f"Your password reset OTP is: {otp}\\\\n"
                                    "It expires in 5 minutes. Do not share this code."
                                )

                                with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
                                    server.login(
                                        os.environ["EMAIL_ADDRESS"],
                                        os.environ["EMAIL_PASSWORD"]
                                    )
                                    server.send_message(msg)

                                st.session_state.reset_email = email
                                st.session_state.reset_mode = "otp"
                                st.session_state.otp_code = otp
                                st.session_state.otp_expiry = time.time() + 300
                                st.session_state.otp_verified = False
                                st.success("OTP sent. Check your email inbox and spam folder.")
                                st.rerun()
                            except Exception:
                                st.error("Could not send OTP. Check Gmail App Password and try again.")'''

if old_otp_button in code:
    code = code.replace(old_otp_button, new_otp_button, 1)
elif 'st.session_state.reset_mode = "otp"' not in code:
    raise RuntimeError("Could not find the original OTP placeholder. No OTP patch applied.")

# Add OTP verification and password reset to the existing recovery page.
old_security_branch = '''                if st.session_state.get("reset_mode") == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")'''

new_security_branch = '''                if st.session_state.get("reset_mode") == "otp":
                    st.subheader("Verify email OTP")
                    entered_otp = st.text_input("6-digit OTP", max_chars=6)
                    if st.button("Verify OTP"):
                        if time.time() > st.session_state.otp_expiry:
                            st.session_state.otp_code = None
                            st.session_state.otp_verified = False
                            st.error("OTP expired. Cancel and start again.")
                        elif secrets.compare_digest(
                            entered_otp, st.session_state.otp_code or ""
                        ):
                            st.session_state.otp_verified = True
                            st.success("OTP verified. You can now set a new password.")
                        else:
                            st.error("Incorrect OTP.")

                    if st.session_state.otp_verified:
                        otp_new_password = st.text_input("New password", type="password", key="otp_new_password")
                        otp_confirm_password = st.text_input("Confirm new password", type="password", key="otp_confirm_password")
                        if st.button("Reset Password", key="otp_reset_password"):
                            if time.time() > st.session_state.otp_expiry:
                                st.session_state.otp_verified = False
                                st.error("OTP expired. Please start again.")
                            elif not valid_password(otp_new_password):
                                st.error("Password must be 8+ characters with uppercase, lowercase, number, and special symbol.")
                            elif otp_new_password != otp_confirm_password:
                                st.error("Passwords do not match.")
                            else:
                                with get_db() as c:
                                    c.execute(
                                        "UPDATE users SET password_hash=? WHERE email=?",
                                        (hash_txt(otp_new_password), st.session_state.reset_email)
                                    )
                                st.session_state.otp_code = None
                                st.session_state.otp_expiry = None
                                st.session_state.otp_verified = False
                                st.session_state.reset_email = None
                                st.session_state.reset_mode = None
                                st.success("Password updated. Please sign in.")
                                time.sleep(1)
                                navigate("Login")

                elif st.session_state.get("reset_mode") == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")'''

if old_security_branch in code:
    code = code.replace(old_security_branch, new_security_branch, 1)
elif 'st.subheader("Verify email OTP")' not in code:
    raise RuntimeError("Could not find the recovery-page insertion point.")

# Clear OTP state when the user cancels recovery.
old_cancel = '''                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                navigate("Login")'''
new_cancel = '''                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.otp_code = None
                st.session_state.otp_expiry = None
                st.session_state.otp_verified = False
                navigate("Login")'''

if old_cancel in code:
    code = code.replace(old_cancel, new_cancel, 1)

# Validate before saving.
ast.parse(code)
path.write_text(code)

print("✅ OTP email sending and verification added to app.py.")
print("✅ OTP expires after 5 minutes.")
print("✅ Password reset requires OTP verification.")
print("✅ Python syntax check passed.")
print("⚠️ OTP delivery still needs a live Gmail test.")
